# Train and segment a tiled area

This notebook walks through a tiled end-to-end tutorial:

1. build tiles from the input rasters
2. attach training crowns to the intersecting tiles
3. build SAM training chips
4. train a SAM decoder
5. inspect SAM proposal outputs
6. build StarDist training records
7. train StarDist
8. run segmentation across all prepared tiles

This is a tutorial for understanding and exercising the tiled workflow.
It is not a recommended evaluation setup.
For real model assessment, keep separate training and held-out areas.


## Inputs

This notebook assumes the clipped rasters are already in the repo `data/` directory.

You still need to provide a crown-label file for training.
That label file can cover only part of the tiled area.


In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import rasterio
from shapely.geometry import box


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "samstars").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from samstars.data import build_scale_stats_from_proposal_records, build_tiles
from samstars.segmentation import run_segmentation
from samstars.training.chips import build_training_chips
from samstars.training.records import build_stardist_training_records
from samstars.training.sam_training import train_decoder
from samstars.training.stardist_training import train_stardist_model


/Users/iosefa/miniforge3/envs/samstars/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


## Set paths

Update the label and checkpoint paths for your own setup.


In [3]:
data_dir = repo_root / "data"

img = data_dir / "img_clipped.tif"
chm = data_dir / "chm_clipped.tif"
den = data_dir / "den_clipped.tif"
crowns = data_dir / "crowns_training.gpkg"
sam_base_ckpt = repo_root / "sam/sam_vit_l_0b3195.pth"
work_dir = data_dir / "train_and_segment_tiled_area_run"
device = "mps"

tiles_dir = work_dir / "tiles"
training_chips_dir = work_dir / "training_chips"
training_run_dir = work_dir / "training_run"
records_dir = work_dir / "stardist_training_records"
models_dir = work_dir / "models"
inference_dir = work_dir / "segmentation_run"
sam_ckpt = models_dir / "sam_crowns_decoder.pth"
stardist_model = models_dir / "stardist"

print("repo_root:", repo_root)
print("data_dir:", data_dir)


repo_root: /Users/iosefa/repos/sam
data_dir: /Users/iosefa/repos/sam/data


## Set tutorial parameters

These are explicit on purpose. Do not treat them as fixed project defaults. Change them, rerun the notebook, and compare the intermediate proposal outputs before committing to StarDist training.

`rgb_bands` controls which three image bands are written into each tile. For a 3-band raster, keep `(1, 2, 3)`. For a multiband raster, choose the three bands you want the full workflow to use.


In [4]:
rgb_bands = (1, 2, 3)

tile_size_m = 500.0
tile_overlap_m = 20.0

chip_size = 512
chip_stride = 384

sam_epochs = 3
stardist_epochs = 10
val_fraction = 0.2

crop_smooth_sigma = 2.0
smooth_sigma = 4.0
post_smooth_sigma = 0.0

stardist_input_key = "index"
use_aux = True
fast_mode = False


## Build tiles

This creates the prepared tile layout used by the tiled workflow.


In [5]:
build_tiles(
    img=img,
    chm=chm,
    den=den,
    rgb_bands=rgb_bands,
    out=tiles_dir,
    tile=tile_size_m,
    overlap=tile_overlap_m,
    skip_existing=True,
)


Tiling:   0%|                                                                  | 0/4 [00:00<?, ?tile/s]function_base.py (4824): Warning: 'partition' will ignore the 'mask' of the MaskedArray.
cost_surface.py (137): No SLIC provided – cost built from 3 terms only.
_base.py (248): Precomputed sparse input was not sorted by row values. Use the function sklearn.neighbors.sort_graph_by_row_values to sort the input by row values, with warn_when_not_sorted=False to remove this warning.
Tiling:  25%|██████████████▌                                           | 1/4 [00:02<00:08,  2.81s/tile]function_base.py (4824): Warning: 'partition' will ignore the 'mask' of the MaskedArray.
cost_surface.py (137): No SLIC provided – cost built from 3 terms only.
_base.py (248): Precomputed sparse input was not sorted by row values. Use the function sklearn.neighbors.sort_graph_by_row_values to sort the input by row values, with warn_when_not_sorted=False to remove this warning.
Tiling:  50%|███████████████████

✓ Tiles written to /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/tiles


## Attach training crowns to intersecting tiles

This clips the training polygons into each tile that overlaps the labeled area.
Only those tiles contribute supervised training data.


In [6]:
crowns_gdf = gpd.read_file(crowns)
labeled_tiles = []

for tile_dir in sorted(p for p in tiles_dir.iterdir() if p.is_dir() and p.name.startswith("tile_")):
    img_path = tile_dir / "img.tif"
    if not img_path.exists():
        continue

    with rasterio.open(img_path) as src:
        tile_bounds = box(*src.bounds)
        tile_crs = src.crs

    tile_crowns = crowns_gdf
    if tile_crowns.crs != tile_crs:
        tile_crowns = tile_crowns.to_crs(tile_crs)

    tile_crowns = tile_crowns[tile_crowns.intersects(tile_bounds)].copy()
    if tile_crowns.empty:
        continue

    tile_crowns["geometry"] = tile_crowns.geometry.buffer(0)
    tile_crowns["geometry"] = tile_crowns.geometry.intersection(tile_bounds)
    tile_crowns = tile_crowns[~tile_crowns.geometry.is_empty]
    if tile_crowns.empty:
        continue

    out_path = tile_dir / "crowns.gpkg"
    tile_crowns.to_file(out_path, driver="GPKG")
    labeled_tiles.append(tile_dir.name)

print("labeled tiles:", labeled_tiles)
print("count:", len(labeled_tiles))


labeled tiles: ['tile_000_001', 'tile_001_001']
count: 2


## Build SAM training chips


In [7]:
build_training_chips(
    tiles_dir=tiles_dir,
    out_dir=training_chips_dir,
    chip_size=chip_size,
    stride=chip_stride,
)


Skipping tile_000_000: missing img.tif, crowns.gpkg, or chm.tif
tile_000_001: wrote 153 training chips
Skipping tile_001_000: missing img.tif, crowns.gpkg, or chm.tif
tile_001_001: wrote 8 training chips
Built 161 training chips in /Users/iosefa/repos/sam/data/train_and_segment_tiled_area_run/training_chips


161

## Train models

This section first trains the SAM decoder, then creates and inspects proposal rasters, and only then trains StarDist.


In [ ]:
train_decoder(
    chip_dir=training_chips_dir,
    ckpt=sam_base_ckpt,
    out_path=sam_ckpt,
    epochs=sam_epochs,
    device=device,
)

training_chips: epochs=3 (iters/epoch=161)


In [ ]:
proposal_result = run_segmentation(
    tiles_dir=tiles_dir,
    out_dir=training_run_dir,
    sam_ckpt=sam_ckpt,
    device=device,
    run_sam=True,
    run_stardist=False,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    fast=fast_mode,
)

## Inspect SAM proposal outputs

Inspect the proposal rasters before continuing.
If the proposals are poor, adjust the parameters above and rerun this section before training StarDist.


In [ ]:
proposal_result


In [ ]:
proposal_rasters_dir = proposal_result["proposals_root"] / "rasters"
sorted(p.name for p in proposal_rasters_dir.glob("*.tif"))[:40]


## Build StarDist training records


In [ ]:
train_records, val_records = build_stardist_training_records(
    proposal_records=proposal_result["proposal_records"],
    tiles_dir=tiles_dir,
    out_dir=records_dir,
    val_fraction=val_fraction,
)

print("train records:", len(__import__('json').load(open(train_records))))
print("val records:", len(__import__('json').load(open(val_records))))


In [ ]:
scale_stats = work_dir / "training_scale_stats.json"

build_scale_stats_from_proposal_records(
    proposal_records=proposal_result["proposal_records"],
    out=scale_stats,
    percentiles=(2.0, 98.0),
    chm_key="chm",
    nodata_values=(-9999.0,),
)


In [ ]:
train_stardist_model(
    train_records=train_records,
    val_records=val_records,
    out_dir=stardist_model.parent,
    epochs=stardist_epochs,
    input_key=stardist_input_key,
    use_aux=use_aux,
    scale_stats_path=scale_stats,
)


## Run tiled segmentation

This runs inference across all prepared tiles.


In [ ]:
result = run_segmentation(
    tiles_dir=tiles_dir,
    out_dir=inference_dir,
    sam_ckpt=sam_ckpt,
    stardist_model=stardist_model,
    stardist_input_key=stardist_input_key,
    device=device,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    fast=fast_mode,
)


## Final output


In [ ]:
inference_dir / "crowns" / "crowns.gpkg"
